# 🇮🇩 RAG Chatbot Ekonomi Indonesia

**Stack:** Python · LangChain · ChromaDB · IndoBERT · LLaMA 3.3-70B (Groq) · NewsAPI · Gradio

**Jalankan cell secara berurutan dari atas ke bawah.**

## Fase 1 — Setup Environment

### Cell 1 · Install Dependencies

In [ ]:
# Verifikasi torch bawaan Colab
import torch
print(f"torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

# Install semua
!pip install sentence-transformers==4.1.0 transformers==4.52.4 -q
!pip install langchain==0.3.25 langchain-community==0.3.24 \
             langchain-groq==0.3.2 langchain-huggingface==0.2.0 -q
!pip install chromadb==0.6.3 -q
!pip install pypdf==5.6.0 python-docx==1.1.2 tiktoken==0.9.0 -q
!pip install requests newsapi-python==0.2.7 -q
!pip install gradio==5.33.0 python-dotenv==1.1.0 tqdm==4.67.1 -q

# Verifikasi
import sentence_transformers, langchain, chromadb
print(f"sentence-transformers : {sentence_transformers.__version__}")
print(f"langchain             : {langchain.__version__}")
print(f"chromadb              : {chromadb.__version__}")
print("\n✅ Siap lanjut!")

### Cell 2 · Mount Google Drive & Struktur Folder

In [ ]:
from google.colab import drive
import os

# Mount Google Drive ke path /content/drive/
drive.mount('/content/drive')
print("✅ Google Drive berhasil di-mount!")

# --- Definisikan semua path sebagai konstanta ---
BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"

PATHS = {
    "base":        BASE_DIR,
    "pdfs":        f"{BASE_DIR}/data/pdfs",
    "vectorstore": f"{BASE_DIR}/vectorstore",
    "models":      f"{BASE_DIR}/models",
    "logs":        f"{BASE_DIR}/logs",
}

# --- Buat semua folder sekaligus ---
for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"📁 {name:12s} → {path}")

print("\n✅ Struktur folder siap!")

### Cell 3 · Buat File .env

> ⚠️ Setelah cell ini dijalankan, buka file `.env` di Google Drive dan isi `GROQ_API_KEY`, `NEWS_API_KEY`, dan `BPS_API_KEY` sebelum lanjut.

In [ ]:
env_content = """# ================================================
# RAG Ekonomi Indonesia — Environment Variables
# JANGAN commit file ini ke GitHub!
# ================================================

# Groq API (untuk LLaMA 3.3-70B)
GROQ_API_KEY=gsk_isi_dengan_groq_api_key_kamu

# NewsAPI (untuk berita real-time)
NEWS_API_KEY=isi_dengan_newsapi_key_kamu

# Path Configuration
VECTORSTORE_PATH=/content/drive/MyDrive/rag-ekonomi-indonesia/vectorstore
PDF_PATH=/content/drive/MyDrive/rag-ekonomi-indonesia/data/pdfs
MODEL_CACHE_PATH=/content/drive/MyDrive/rag-ekonomi-indonesia/models
"""

env_path = f"{BASE_DIR}/.env"
if not os.path.exists(env_path):
    with open(env_path, 'w') as f:
        f.write(env_content)
    print(f"✅ File .env dibuat di: {env_path}")
else:
    print(f"ℹ️  File .env sudah ada — tidak ditimpa")
    print("⚠️  Edit file .env di Google Drive dan isi semua API key sebelum lanjut!")

### Cell 4 · Verifikasi Konfigurasi

In [ ]:
import os
from dotenv import load_dotenv

# --- Load .env dari Google Drive ---
BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
env_path = f"{BASE_DIR}/.env"

load_dotenv(env_path)
print(f"📄 Membaca .env dari: {env_path}")

# --- Ambil semua key ---
GROQ_API_KEY   = os.getenv("GROQ_API_KEY")
NEWS_API_KEY   = os.getenv("NEWS_API_KEY")
VECTORSTORE_PATH = os.getenv("VECTORSTORE_PATH")
PDF_PATH         = os.getenv("PDF_PATH")
MODEL_CACHE_PATH = os.getenv("MODEL_CACHE_PATH")

# --- Validasi ---
def validate_key(name, value, placeholder_hints=["isi_dengan", "xxx"]):
    if value is None:
        return "❌ TIDAK DITEMUKAN"
    if any(hint in value for hint in placeholder_hints):
        return "⚠️  BELUM DIISI (masih placeholder)"
    masked = value[:6] + "*" * (len(value) - 6)
    return f"✅ OK → {masked}"

print("\n🔑 Status API Keys:")
print(f"  GROQ_API_KEY   : {validate_key('GROQ_API_KEY', GROQ_API_KEY)}")
print(f"  NEWS_API_KEY   : {validate_key('NEWS_API_KEY', NEWS_API_KEY)}")

print("\n📁 Status Path Config:")
for name, path in [
    ("VECTORSTORE_PATH", VECTORSTORE_PATH),
    ("PDF_PATH",         PDF_PATH),
    ("MODEL_CACHE_PATH", MODEL_CACHE_PATH),
]:
    exists = "✅ folder ada" if path and os.path.exists(path) else "❌ folder tidak ditemukan"
    print(f"  {name:18s}: {exists}")

# --- Verdict akhir ---
print("\n" + "="*50)
groq_ok = GROQ_API_KEY and "isi_dengan" not in GROQ_API_KEY
news_ok = NEWS_API_KEY and "isi_dengan" not in NEWS_API_KEY

if groq_ok and news_ok:
    print("🚀 Semua konfigurasi valid! Siap lanjut ke Fase 2.")
elif groq_ok and not news_ok:
    print("🟡 Groq OK, tapi NewsAPI belum diisi.")
    print("   Kamu tetap bisa lanjut — NewsAPI dipakai di Fase 4 (real-time retrieval).")
    print("   Daftar di: https://newsapi.org/register")
else:
    print("🔴 GROQ_API_KEY belum diisi — wajib diisi sebelum lanjut!")
    print("   Ambil key di: https://console.groq.com")

## Fase 2 — Embedding & Vector Store

### Cell 5 · Inisialisasi IndoBERT Embedding

In [ ]:
import os
import torch
import numpy as np
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModel
from langchain.embeddings.base import Embeddings
from typing import List

BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
load_dotenv(f"{BASE_DIR}/.env")
MODEL_CACHE_PATH = os.getenv("MODEL_CACHE_PATH")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Hardware: {device.upper()} | {torch.cuda.get_device_name(0)}")

MODEL_NAME = "indobenchmark/indobert-base-p1"
print(f"\n⏳ Memuat IndoBERT dari cache...")

# Load tokenizer & model langsung via transformers
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    cache_dir=MODEL_CACHE_PATH
)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    cache_dir=MODEL_CACHE_PATH,
    use_safetensors=True
).to(device)
model.eval()
print("✅ Model berhasil dimuat!\n")

# --- Buat class embedding custom ---

class IndoBERTEmbeddings(Embeddings):
    """Custom embedding class menggunakan IndoBERT secara langsung."""

    def _mean_pooling(self, model_output, attention_mask):
        """
        Mean pooling: rata-rata semua token jadi satu vektor.
        Kenapa perlu ini? BERT menghasilkan vektor per token (misal 128 token
        = 128 vektor). Kita butuh SATU vektor per kalimat.
        attention_mask dipakai supaya token padding ([PAD]) tidak ikut dirata-rata.
        """
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).float()
        return (token_embeddings * input_mask_expanded).sum(1) / \
               input_mask_expanded.sum(1).clamp(min=1e-9)

    def _embed(self, texts: List[str]) -> List[List[float]]:
        encoded = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():  # tidak perlu hitung gradien saat inference
            output = model(**encoded)

        embeddings = self._mean_pooling(output, encoded["attention_mask"])

        # Normalisasi → panjang vektor = 1, cosine similarity jadi akurat
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        return embeddings.cpu().numpy().tolist()

    def embed_query(self, text: str) -> List[float]:
        """Embed satu query/pertanyaan."""
        return self._embed([text])[0]

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """Embed banyak dokumen sekaligus (untuk indexing)."""
        return self._embed(texts)

# Inisialisasi
embeddings = IndoBERTEmbeddings()
print("✅ IndoBERTEmbeddings siap dipakai!\n")

# --- Tes semantic similarity ---
def cosine_similarity(v1, v2):
    v1, v2 = np.array(v1), np.array(v2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

print("🧪 Tes semantic similarity...")
vec1 = embeddings.embed_query("Bank Indonesia menaikkan suku bunga acuan")
vec2 = embeddings.embed_query("BI mengerek rate bunga acuan naik")
vec3 = embeddings.embed_query("resep nasi goreng spesial")

sim_relevan       = cosine_similarity(vec1, vec2)
sim_tidak_relevan = cosine_similarity(vec1, vec3)

print(f"  'BI naikkan bunga' vs 'BI mengerek rate'  → {sim_relevan:.4f} (harap ≥ 0.80)")
print(f"  'BI naikkan bunga' vs 'resep nasi goreng' → {sim_tidak_relevan:.4f} (harap ≤ 0.50)")

print("\n" + "="*50)
if sim_relevan > 0.75 and sim_tidak_relevan < 0.55:
    print("✅ IndoBERT paham semantik Bahasa Indonesia!")
    print("🚀 Siap lanjut ke Cell 5: Inisialisasi ChromaDB")
else:
    print("⚠️  Hasil di luar ekspektasi — paste output untuk dianalisis")

### Cell 6 · Inisialisasi ChromaDB

In [ ]:
import os
import chromadb
from dotenv import load_dotenv
from chromadb.config import Settings

BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
load_dotenv(f"{BASE_DIR}/.env")
VECTORSTORE_PATH = os.getenv("VECTORSTORE_PATH")

print(f"📂 Vectorstore path: {VECTORSTORE_PATH}")

# --- Inisialisasi ChromaDB persistent client ---
#   EphemeralClient : cepat, tapi hilang saat Colab disconnect
#   PersistentClient: sedikit lebih lambat, tapi data aman di Drive
chroma_client = chromadb.PersistentClient(
    path=VECTORSTORE_PATH,
    settings=Settings(
        anonymized_telemetry=False
    )
)

# --- Buat/load collection ---
# Collection di ChromaDB = "tabel" di database relasional
# get_or_create → kalau sudah ada, load; kalau belum, buat baru
# Aman di-run berkali-kali tanpa duplikasi data

# Kita pakai cosine similarity karena:
# - Vektor kita sudah dinormalisasi (panjang = 1)
# - Cosine mengukur SUDUT antar vektor (kesamaan arah = kesamaan makna)
# - Lebih robust dari euclidean distance untuk teks
COLLECTION_NAME = "ekonomi_indonesia"

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

print(f"✅ Collection '{COLLECTION_NAME}' siap!")
print(f"   Jumlah dokumen saat ini: {collection.count()}")

# --- Tes insert & query sederhana ---

print("\n🧪 Tes insert & similarity search...")

# Data dummy untuk tes
tes_docs = [
    "Bank Indonesia mempertahankan suku bunga acuan di level 6 persen",
    "BPS mencatat inflasi Indonesia sebesar 2.8 persen year on year",
    "Rupiah menguat terhadap dolar Amerika Serikat di pasar valuta asing",
]
tes_ids = ["tes_001", "tes_002", "tes_003"]

# Embed dokumen tes menggunakan IndoBERT kita
tes_vectors = embeddings.embed_documents(tes_docs)

# Insert ke ChromaDB
collection.add(
    embeddings=tes_vectors,
    documents=tes_docs,
    metadatas=[{"sumber": "tes", "topik": "ekonomi"} for _ in tes_docs],
    ids=tes_ids
)
print(f"   ✅ {len(tes_docs)} dokumen tes berhasil diinsert")
print(f"   Total dokumen di collection: {collection.count()}")

# Query: cari dokumen paling relevan untuk pertanyaan ini
query = "bagaimana kondisi nilai tukar rupiah?"
query_vector = embeddings.embed_query(query)

results = collection.query(
    query_embeddings=[query_vector],
    n_results=2,  # ambil 2 dokumen paling relevan
    include=["documents", "distances", "metadatas"]
)

print(f"\n   Query: '{query}'")
print(f"   Hasil retrieval:")
for i, (doc, dist) in enumerate(zip(
    results["documents"][0],
    results["distances"][0]
)):
    similarity = 1 - (dist / 2)
    print(f"   [{i+1}] similarity={similarity:.4f} | {doc[:70]}...")

# --- Bersihkan data tes ---
collection.delete(ids=tes_ids)
print(f"\n   🧹 Data tes dihapus — collection kembali kosong")
print(f"   Jumlah dokumen sekarang: {collection.count()}")

print("\n" + "="*50)
print("✅ ChromaDB berfungsi sempurna!")
print("🚀 Siap lanjut ke Cell 6: PDF Ingestion Pipeline")

## Fase 3 — PDF Ingestion Pipeline

### Cell 7 · Ingestion PDF ke ChromaDB

> 💡 Taruh file PDF dari BI/BPS/Kemenkeu di folder `data/pdfs/` sebelum menjalankan cell ini. Cell ini aman dijalankan berkali-kali — dokumen yang sudah ada tidak akan diduplikasi.

In [ ]:
import os
import glob
import hashlib
from dotenv import load_dotenv
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
load_dotenv(f"{BASE_DIR}/.env")
PDF_PATH = os.getenv("PDF_PATH")

# --- Konfigurasi chunking ---

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

def load_and_chunk_pdf(pdf_path: str) -> list:
    """
    Load satu PDF dan pecah jadi chunks.
    Return list of LangChain Document objects, masing-masing berisi:
    - page_content : teks chunk
    - metadata     : sumber file, nomor halaman
    """
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()

    # Split semua halaman sekaligus
    chunks = text_splitter.split_documents(pages)

    # Tambah metadata tambahan yang berguna saat retrieval nanti
    filename = os.path.basename(pdf_path)
    for i, chunk in enumerate(chunks):
        chunk.metadata.update({
            "filename": filename,
            "chunk_index": i,
            "total_chunks": len(chunks)
        })

    return chunks

def ingest_pdfs(pdf_dir: str, collection, embeddings_model) -> dict:
    """
    Load semua PDF dari folder, chunk, embed, dan simpan ke ChromaDB.
    Return statistik hasil ingestion.
    """
    pdf_files = glob.glob(os.path.join(pdf_dir, "*.pdf"))

    if not pdf_files:
        print(f"⚠️  Tidak ada PDF di: {pdf_dir}")
        print(f"   Taruh file PDF dari BI/BPS/Kemenkeu di folder tersebut,")
        print(f"   lalu jalankan cell ini lagi.")
        return {"total_files": 0, "total_chunks": 0}

    print(f"📄 Ditemukan {len(pdf_files)} file PDF:")
    for f in pdf_files:
        size_mb = os.path.getsize(f) / 1_000_000
        print(f"   - {os.path.basename(f)} ({size_mb:.1f} MB)")

    all_chunks = []
    stats = {"total_files": len(pdf_files), "total_chunks": 0, "per_file": {}}

    # --- Proses setiap PDF ---
    for pdf_path in tqdm(pdf_files, desc="📥 Loading PDFs"):
        filename = os.path.basename(pdf_path)
        try:
            chunks = load_and_chunk_pdf(pdf_path)
            all_chunks.extend(chunks)
            stats["per_file"][filename] = len(chunks)
            print(f"   ✅ {filename}: {len(chunks)} chunks")
        except Exception as e:
            print(f"   ❌ {filename}: gagal — {e}")

    if not all_chunks:
        return stats

    # --- Embed dan simpan ke ChromaDB ---
    BATCH_SIZE = 32
    total_batches = (len(all_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

    print(f"\n⚡ Embedding {len(all_chunks)} chunks dalam {total_batches} batch...")

    for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="🔢 Embedding"):
        batch = all_chunks[i:i + BATCH_SIZE]

        texts     = [c.page_content for c in batch]
        metadatas = [c.metadata for c in batch]
        ids = [
    hashlib.md5(
        f"{meta.get('filename','')}-{meta.get('chunk_index','')}-{text[:100]}"
        .encode()
    ).hexdigest()[:16]
    for text, meta in zip(texts, metadatas)
]
        vectors   = embeddings_model.embed_documents(texts)

        collection.upsert(
        embeddings=vectors,
        documents=texts,
        metadatas=metadatas,
        ids=ids
)

    stats["total_chunks"] = len(all_chunks)
    return stats

# --- Jalankan ingestion ---
print("="*50)
print("🚀 Memulai PDF Ingestion Pipeline")
print("="*50)

stats = ingest_pdfs(PDF_PATH, collection, embeddings)

print(f"\n📊 Hasil Ingestion:")
print(f"   Total file    : {stats['total_files']}")
print(f"   Total chunks  : {stats['total_chunks']}")
print(f"   Di ChromaDB   : {collection.count()} dokumen")

if stats["total_chunks"] > 0:
    print("\n✅ Ingestion selesai! Data tersimpan permanen di Google Drive.")
    print("🚀 Siap lanjut ke Cell 7: NewsAPI Real-time Retrieval")

## Fase 4 — NewsAPI Real-time Retrieval

### Cell 8 · Cek NewsAPI Key

In [ ]:
import os
from dotenv import load_dotenv

BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
load_dotenv(f"{BASE_DIR}/.env")

NEWS_API_KEY = os.getenv("NEWS_API_KEY")
placeholder = NEWS_API_KEY and "isi_dengan" not in NEWS_API_KEY

print(f"NEWS_API_KEY: {'✅ Sudah diisi' if placeholder else '⚠️  Belum diisi'}")

### Cell 9 · Fungsi Fetch Berita Ekonomi

In [ ]:
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv
from newsapi import NewsApiClient

BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
load_dotenv(f"{BASE_DIR}/.env")
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

newsapi = NewsApiClient(api_key=NEWS_API_KEY)

# --- Kata kunci ekonomi untuk filter relevansi ---
EKONOMI_KEYWORDS = [
    "ekonomi", "inflasi", "rupiah", "bank indonesia", "bi rate",
    "gdp", "pdb", "ekspor", "impor", "investasi", "saham", "ihsg",
    "suku bunga", "devisa", "apbn", "fiskal", "moneter", "bps",
    "pertumbuhan", "kemenkeu", "ojk", "neraca", "perdagangan",
    "kurs", "dolar", "valas", "obligasi", "sbsn", "sri", "yield",
]

def extract_search_keywords(query: str) -> str:
    """
    Sederhanakan query panjang jadi 2-3 kata kunci untuk NewsAPI.
    NewsAPI bekerja lebih baik dengan keywords spesifik
    daripada kalimat panjang penuh stopwords.
    """
    stopwords = [
        "apa", "itu", "dan", "atau", "yang", "untuk", "dari", "ke",
        "di", "dengan", "adalah", "bagaimana", "mengapa", "kenapa",
        "berapa", "kapan", "siapa", "jelaskan", "ceritakan", "tolong",
        "penting", "bagi", "tentang", "mengenai", "terkait", "mohon",
        "apakah", "sebutkan", "kondisi", "situasi", "update", "berita",
        "kabar", "terbaru", "terkini", "saat", "ini", "sekarang",
    ]
    words = query.lower().split()
    keywords = [w for w in words if w not in stopwords and len(w) > 3]
    core = " ".join(keywords[:3])
    if "indonesia" not in core.lower():
        core += " Indonesia"
    return core

def is_ekonomi_relevant(article: dict) -> bool:
    """Cek apakah artikel benar-benar tentang ekonomi Indonesia."""
    text = (article["title"] + " " + article["content"]).lower()
    return any(kw in text for kw in EKONOMI_KEYWORDS)

def fetch_economic_news(query: str, max_articles: int = 5) -> list:
    """
    Ambil berita ekonomi Indonesia relevan dari NewsAPI.
    Selalu return list — tidak pernah return None.
    """
    date_from = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    keywords  = extract_search_keywords(query)

    # --- Coba query spesifik dulu ---
    try:
        response = newsapi.get_everything(
            q=f"{keywords} ekonomi",
            from_param=date_from,
            sort_by="relevancy",
            page_size=max_articles * 2
        )
    except Exception as e:
        print(f"❌ NewsAPI error: {e}")
        return []

    if response["status"] != "ok":
        print(f"❌ NewsAPI response error: {response.get('message', 'unknown')}")
        return []

    articles = response.get("articles", [])

    # --- Fallback ke query umum kalau hasil kosong ---
    if not articles:
        print(f"⚠️  Tidak ada hasil untuk '{keywords}', coba query fallback...")
        try:
            response = newsapi.get_everything(
                q="ekonomi Indonesia",
                from_param=date_from,
                sort_by="publishedAt",
                page_size=max_articles * 2
            )
            articles = response.get("articles", [])
        except Exception as e:
            print(f"❌ Fallback error: {e}")
            return []

    if not articles:
        return []

    # --- Proses, filter, dan return ---
    results = []
    for article in articles:
        title       = article.get("title", "")       or ""
        description = article.get("description", "") or ""
        content     = article.get("content", "")     or ""

        full_text = f"{title}. {description} {content}".strip()
        full_text = " ".join(full_text.split())

        if len(full_text) < 50:
            continue

        article_dict = {
            "title":        title,
            "content":      full_text,
            "url":          article.get("url", ""),
            "published_at": article.get("publishedAt", ""),
            "source":       article.get("source", {}).get("name", "Unknown"),
        }

        # Hanya simpan artikel yang benar-benar tentang ekonomi
        if is_ekonomi_relevant(article_dict):
            results.append(article_dict)

        # Berhenti kalau sudah cukup
        if len(results) >= max_articles:
            break

    return results

def format_news_context(articles: list) -> str:
    """Format artikel jadi string konteks untuk LLM."""
    if not articles:
        return "Tidak ada berita ekonomi terkini yang ditemukan."

    context_parts = []
    for i, article in enumerate(articles, 1):
        try:
            dt      = datetime.fromisoformat(article["published_at"].replace("Z", "+00:00"))
            tanggal = dt.strftime("%d %B %Y")
        except Exception:
            tanggal = article["published_at"][:10] if article["published_at"] else "Tanggal tidak diketahui"

        context_parts.append(
            f"[Berita {i}] {article['source']} — {tanggal}\n"
            f"Judul: {article['title']}\n"
            f"Isi: {article['content']}\n"
            f"URL: {article['url']}"
        )

    return "\n\n".join(context_parts)

# --- Tes ---
print("🧪 Tes NewsAPI Retrieval (Final)\n")

test_queries = ["inflasi", "nilai tukar rupiah", "BI Rate suku bunga"]

for query in test_queries:
    articles = fetch_economic_news(query, max_articles=2)
    print(f"Query: '{query}'")
    print(f"  → {len(articles)} artikel relevan ditemukan")
    if articles:
        print(f"  → Terbaru : {articles[0]['title'][:65]}...")
        print(f"  → Sumber  : {articles[0]['source']}")
    print()

print("📋 Contoh format konteks untuk LLM:")
print("-" * 50)
sample = fetch_economic_news("inflasi Indonesia", max_articles=2)
print(format_news_context(sample)[:600] + "...")

print("\n" + "="*50)
print("✅ NewsAPI Cell 7 final — siap dipakai!")

## Fase 5 — Query Router

### Cell 10 · Query Router & Retrieve Context

In [ ]:
import re
from enum import Enum

class QueryRoute(Enum):
    CHROMADB = "chromadb"
    NEWSAPI  = "newsapi"

def route_query(query: str) -> tuple[QueryRoute, str]:
    """
    Tentukan jalur retrieval untuk query yang diberikan.

    Return:
        (QueryRoute, alasan)  ← alasan berguna untuk debugging & logging
    """
    query_lower = query.lower().strip()

    # --- Kata kunci yang menandakan butuh informasi real-time ---
    temporal_keywords = [
        # Waktu eksplisit
        "hari ini", "kemarin", "minggu ini", "bulan ini", "tahun ini",
        "pekan ini", "tadi", "baru saja", "terkini", "terbaru",
        "sekarang", "saat ini", "kini", "terkini",
        # Kata kerja yang implisit real-time
        "naik", "turun", "menguat", "melemah", "anjlok", "meroket",
        "reaksi", "respon", "dampak", "efek",
        # Referensi waktu relatif
        "2025", "2026", "q1", "q2", "q3", "q4",
        "januari", "februari", "maret", "april", "mei", "juni",
        "juli", "agustus", "september", "oktober", "november", "desember",
        # Konteks berita
        "berita", "kabar", "update", "perkembangan", "kondisi",
        "prediksi", "proyeksi", "forecast",
    ]

    # --- Kata kunci yang menandakan butuh dokumen referensi ---
    static_keywords = [
        "apa itu", "jelaskan", "pengertian", "definisi",
        "bagaimana cara", "mekanisme", "kebijakan", "regulasi",
        "undang-undang", "peraturan", "sejarah", "historis",
    ]

    # Hitung skor untuk setiap jalur
    temporal_score = sum(1 for kw in temporal_keywords if kw in query_lower)
    static_score   = sum(1 for kw in static_keywords   if kw in query_lower)

    # Logika keputusan
    if temporal_score > static_score:
        return QueryRoute.NEWSAPI, f"temporal_score={temporal_score} > static_score={static_score}"
    elif static_score > temporal_score:
        return QueryRoute.CHROMADB, f"static_score={static_score} > temporal_score={temporal_score}"
    else:
        return QueryRoute.CHROMADB, "no clear signal → default ke ChromaDB"

def retrieve_context(query: str, collection, embeddings_model,
                     n_results: int = 3) -> tuple[str, QueryRoute]:
    """
    Fungsi utama retrieval: route query → ambil konteks → return teks.

    Return:
        (context_string, route_yang_dipakai)
    """
    route, reason = route_query(query)
    print(f"   🔀 Route: {route.value} ({reason})")

    if route == QueryRoute.NEWSAPI:
        articles = fetch_economic_news(query, max_articles=n_results)
        context  = format_news_context(articles)
        return context, route

    else:  # ChromaDB
        if collection.count() == 0:
            print(f"   ⚠️  ChromaDB kosong → fallback ke NewsAPI")
            articles = fetch_economic_news(query, max_articles=n_results)
            context  = format_news_context(articles)
            return context, QueryRoute.NEWSAPI

        query_vector = embeddings_model.embed_query(query)
        results = collection.query(
            query_embeddings=[query_vector],
            n_results=min(n_results, collection.count()),
            include=["documents", "distances", "metadatas"]
        )

        # Format hasil ChromaDB jadi string konteks
        context_parts = []
        for i, (doc, dist, meta) in enumerate(zip(
            results["documents"][0],
            results["distances"][0],
            results["metadatas"][0]
        ), 1):
            similarity = 1 - (dist / 2)
            source     = meta.get("filename", "Dokumen")
            page       = meta.get("page", "?")
            context_parts.append(
                f"[Dokumen {i}] {source} — Hal. {page} "
                f"(similarity: {similarity:.3f})\n{doc}"
            )

        context = "\n\n".join(context_parts)
        return context, route

# --- Tes Query Router ---
print("🧪 Tes Query Router\n")

test_queries = [
    "Apa itu inflasi dan bagaimana cara mengukurnya?",
    "Bagaimana kondisi rupiah hari ini?",
    "Jelaskan mekanisme transmisi kebijakan moneter BI",
    "Inflasi bulan Juni 2026 berapa persen?",
    "Apa pengertian cadangan devisa?",
    "Update terbaru nilai tukar rupiah terhadap dolar",
]

for query in test_queries:
    route, reason = route_query(query)
    icon = "📰" if route == QueryRoute.NEWSAPI else "📚"
    print(f"{icon} [{route.value:8s}] {query}")

print("\n" + "="*50)
print("✅ Query Router siap!")
print("🚀 Siap lanjut ke Cell 9: LLM Chain dengan Groq")

## Fase 6 — LLM Chain

### Cell 11 · LLM Chain dengan Groq + Fallback

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage

BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
load_dotenv(f"{BASE_DIR}/.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# --- Konfigurasi model dengan fallback chain ---
MODELS = [
    {"name": "llama-3.3-70b-versatile",      "label": "LLaMA 3.3-70B"},
    {"name": "qwen-qwq-32b",                 "label": "Qwen3-32B"},
    {"name": "llama-3.1-8b-instant",         "label": "LLaMA 3.1-8B"},
    {"name": "gemma2-9b-it",                 "label": "Gemma2-9B"},
]

def get_llm(model_config: dict) -> ChatGroq:
    """Inisialisasi LLM dari konfigurasi model."""
    return ChatGroq(
        api_key=GROQ_API_KEY,
        model=model_config["name"],
        temperature=0.2,
        max_tokens=1024,
    )

# --- System prompt ---
SYSTEM_PROMPT = """Kamu adalah asisten ekonomi Indonesia yang berpengetahuan luas.
Tugasmu adalah menjawab pertanyaan tentang ekonomi Indonesia berdasarkan konteks yang diberikan.

Aturan:
1. Jawab HANYA berdasarkan konteks yang diberikan — jangan mengarang fakta
2. Jika konteks tidak cukup untuk menjawab, katakan dengan jelas
3. Gunakan Bahasa Indonesia yang formal namun mudah dipahami
4. Sertakan angka/data spesifik jika tersedia di konteks
5. Sebutkan sumber informasi (nama dokumen atau media) di akhir jawaban
6. Jawaban maksimal 3-4 paragraf — padat dan informatif"""

def build_prompt(query: str, context: str, route: QueryRoute) -> str:
    """
    Susun prompt lengkap untuk LLM.
    Struktur prompt yang baik = jawaban yang baik.
    """
    source_label = "Berita Terkini" if route == QueryRoute.NEWSAPI \
                   else "Dokumen Referensi"

    return f"""Berdasarkan {source_label} berikut, jawab pertanyaan pengguna.

=== {source_label} ===
{context}

=== Pertanyaan ===
{query}

=== Jawaban ==="""

def ask(query: str, verbose: bool = True) -> dict:
    """
    Fungsi utama chatbot — dari query sampai jawaban.

    Return dict berisi: answer, route, model_used, context
    """
    if verbose:
        print(f"❓ Query: {query}")
        print(f"{'─'*50}")

    # Step 1: Route & Retrieve
    context, route = retrieve_context(
        query, collection, embeddings,
        n_results=3
    )

    # Step 2: Build prompt
    prompt = build_prompt(query, context, route)

    # Step 3: Generate dengan fallback chain
    answer       = None
    model_used   = None
    last_error   = None

    for model_config in MODELS:
        try:
            if verbose:
                print(f"   🤖 Mencoba {model_config['label']}...")

            llm = get_llm(model_config)
            messages = [
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=prompt)
            ]
            response   = llm.invoke(messages)
            answer     = response.content
            model_used = model_config["label"]
            break  # berhasil — keluar dari loop fallback

        except Exception as e:
            last_error = str(e)
            if verbose:
                print(f"   ⚠️  {model_config['label']} gagal: {str(e)[:60]}...")
            continue

    if answer is None:
        answer     = f"Maaf, semua model tidak tersedia saat ini. Error: {last_error}"
        model_used = "none"

    if verbose:
        print(f"   ✅ Dijawab oleh: {model_used}")
        print(f"   📡 Sumber data: {route.value}")
        print(f"\n{'='*50}")
        print(f"💬 Jawaban:\n{answer}")
        print(f"{'='*50}\n")

    return {
        "query":      query,
        "answer":     answer,
        "route":      route.value,
        "model_used": model_used,
        "context":    context,
    }

# --- Tes end-to-end ---
print("🧪 Tes End-to-End RAG Chain\n")

# Tes 1: pertanyaan konseptual → ChromaDB (fallback ke NewsAPI karena DB kosong)
result1 = ask("Apa itu cadangan devisa dan mengapa penting bagi Indonesia?")

print()

# Tes 2: pertanyaan real-time → NewsAPI
result2 = ask("Apa berita terbaru tentang ekonomi Indonesia?")

## Fase 7 — Gradio UI

### Cell 12 · Gradio Chatbot Interface

> Setelah dijalankan, klik link **public URL** yang muncul untuk membuka chatbot di browser.

In [ ]:
import gradio as gr
from datetime import datetime

# --- Fungsi wrapper untuk Gradio ---

def chat(message: str, history: list) -> str:
    """Wrapper fungsi ask() untuk interface Gradio."""
    if not message.strip():
        return "Silakan masukkan pertanyaan tentang ekonomi Indonesia."

    result = ask(message, verbose=False)

    # Tambahkan metadata di bawah jawaban
    route_label = "📰 Berita Real-time" if result["route"] == "newsapi" \
                  else "📚 Dokumen Referensi"
    timestamp   = datetime.now().strftime("%H:%M:%S")

    answer = (
        f"{result['answer']}\n\n"
        f"---\n"
        f"*Sumber: {route_label} · Model: {result['model_used']} · {timestamp}*"
    )
    return answer

# --- Contoh pertanyaan untuk tombol quick-access ---
EXAMPLE_QUESTIONS = [
    "Apa itu inflasi dan bagaimana cara mengukurnya?",
    "Bagaimana kondisi nilai tukar rupiah terkini?",
    "Jelaskan kebijakan moneter Bank Indonesia",
    "Apa berita ekonomi Indonesia terbaru?",
    "Berapa cadangan devisa Indonesia saat ini?",
    "Bagaimana pertumbuhan ekonomi Indonesia tahun ini?",
]

# --- Build UI ---
with gr.Blocks(
    title="RAG Ekonomi Indonesia",
    theme=gr.themes.Soft(),
) as demo:

    # Header
    gr.Markdown("""
    # 🇮🇩 RAG Chatbot Ekonomi Indonesia
    Chatbot berbasis **Retrieval-Augmented Generation (RAG)** untuk menjawab
    pertanyaan seputar ekonomi Indonesia.

    **Sumber data:**
    - 📚 Dokumen resmi BI, BPS, Kemenkeu (via ChromaDB)
    - 📰 Berita ekonomi real-time (via NewsAPI)

    **Model:** LLaMA 3.3-70B via Groq · **Embedding:** IndoBERT
    """)

    # Chat interface
    chatbot = gr.ChatInterface(
        fn=chat,
        type="messages",
        chatbot=gr.Chatbot(
            height=450,
            placeholder="Tanyakan seputar ekonomi Indonesia...",
            show_label=False,
            type="messages",
        ),
        textbox=gr.Textbox(
            placeholder="Contoh: Apa itu inflasi dan bagaimana dampaknya?",
            container=False,
            scale=7,
        ),
        examples=EXAMPLE_QUESTIONS,
        cache_examples=False,
    )

    # Footer
    gr.Markdown("""
    ---
    💡 **Tips:** Tanya tentang kebijakan, data makroekonomi, atau berita terkini.
    Untuk data real-time (hari ini, minggu ini), chatbot otomatis menggunakan sumber berita.

    """)

# --- Launch ---
print("🚀 Meluncurkan Gradio UI...")
print("   Tunggu beberapa detik, lalu klik link yang muncul\n")

demo.launch(
    share=True,        # buat public link (ngrok tunnel) — bisa diakses dari HP
    debug=False,
    show_error=True,
)

## Fase 3.5 — Auto-Scraping PDF (Opsional)

> Jalankan fase ini setelah Gradio berjalan. Scheduler akan otomatis mengupdate ChromaDB setiap 24 jam dengan PDF terbaru dari BPS dan Kemenkeu Fiskal.

### Cell 13 · Install Dependencies Scraping

In [ ]:
!pip install apscheduler==3.10.4 beautifulsoup4==4.13.4 lxml==5.3.2 -q

import bs4, apscheduler
print(f"beautifulsoup4 : {bs4.__version__}")
print(f"apscheduler    : {apscheduler.__version__}")
print("✅ Siap!")

### Cell 14 · Auto-Scraper BPS + Kemenkeu + Scheduler

In [ ]:
import os
import time
import requests
import hashlib
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.interval import IntervalTrigger

BASE_DIR = "/content/drive/MyDrive/rag-ekonomi-indonesia"
load_dotenv(f"{BASE_DIR}/.env")
PDF_PATH = os.getenv("PDF_PATH")

# Header agar tidak diblokir sebagai bot
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

# --- Registry PDF yang sudah didownload ---
# Kita simpan hash filename supaya tidak download ulang
REGISTRY_FILE = f"{BASE_DIR}/logs/downloaded_pdfs.txt"
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)

def load_registry() -> set:
    """Load daftar PDF yang sudah pernah didownload."""
    if not os.path.exists(REGISTRY_FILE):
        return set()
    with open(REGISTRY_FILE, "r") as f:
        return set(line.strip() for line in f if line.strip())

def save_to_registry(filename: str):
    """Catat PDF yang baru saja didownload."""
    with open(REGISTRY_FILE, "a") as f:
        f.write(filename + "\n")

def download_pdf(url: str, filename: str, registry: set) -> bool:
    """
    Download satu PDF kalau belum ada di registry.
    Return True kalau berhasil download, False kalau skip/gagal.
    """
    if filename in registry:
        return False  # sudah ada, skip

    filepath = os.path.join(PDF_PATH, filename)
    if os.path.exists(filepath):
        save_to_registry(filename)
        return False  # file sudah ada di disk

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        # Validasi: pastikan response benar-benar PDF
        content_type = response.headers.get("Content-Type", "")
        if "pdf" not in content_type.lower() and not url.endswith(".pdf"):
            return False

        with open(filepath, "wb") as f:
            f.write(response.content)

        save_to_registry(filename)
        size_kb = len(response.content) / 1024
        print(f"      ✅ Downloaded: {filename} ({size_kb:.0f} KB)")
        return True

    except Exception as e:
        print(f"      ❌ Gagal download {filename}: {str(e)[:60]}")
        return False

# ============================================================
# SCRAPER 1: BPS — Berita Resmi Statistik
# ============================================================
def scrape_bps(registry: set) -> int:
    """Scrape PDF Berita Resmi Statistik dari BPS."""
    print("   📊 Scraping BPS...")
    url      = "https://www.bps.go.id/id/pressrelease"
    new_count = 0

    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, "lxml")

        # BPS: cari semua link yang mengarah ke PDF atau halaman detail BRS
        links = soup.find_all("a", href=True)
        pdf_links = []

        for link in links:
            href = link["href"]
            # BPS menyimpan PDF di path /id/pressrelease/download/...
            if "/pressrelease" in href and ("download" in href or href.endswith(".pdf")):
                if not href.startswith("http"):
                    href = "https://www.bps.go.id" + href
                pdf_links.append(href)

        # Ambil maksimal 10 PDF terbaru
        for pdf_url in pdf_links[:10]:
            # Generate filename dari URL
            filename = "bps_" + hashlib.md5(pdf_url.encode()).hexdigest()[:8] + ".pdf"
            if download_pdf(pdf_url, filename, registry):
                new_count += 1
                registry.add(filename)
                time.sleep(1)  # jeda 1 detik antar download

    except Exception as e:
        print(f"      ❌ BPS scraping error: {str(e)[:80]}")

    print(f"      → {new_count} PDF baru dari BPS")
    return new_count

# ============================================================
# SCRAPER 2: Bank Indonesia — Laporan & Publikasi
# ============================================================
def scrape_bi(registry: set) -> int:
    """Scrape PDF publikasi dari Bank Indonesia."""
    print("   🏦 Scraping Bank Indonesia...")
    new_count = 0

    # Halaman-halaman publikasi BI yang relevan
    bi_pages = [
        "https://www.bi.go.id/id/publikasi/laporan/Pages/default.aspx",
        "https://www.bi.go.id/id/publikasi/kajian-ekonomi-regional/Pages/default.aspx",
    ]

    for page_url in bi_pages:
        try:
            resp = requests.get(page_url, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(resp.text, "lxml")

            # BI: cari link PDF langsung
            for link in soup.find_all("a", href=True):
                href = link["href"]
                if href.lower().endswith(".pdf"):
                    if not href.startswith("http"):
                        href = "https://www.bi.go.id" + href

                    filename = "bi_" + hashlib.md5(href.encode()).hexdigest()[:8] + ".pdf"
                    if download_pdf(href, filename, registry):
                        new_count += 1
                        registry.add(filename)
                        time.sleep(1)

                    if new_count >= 5:  # maks 5 per sesi
                        break

        except Exception as e:
            print(f"      ❌ BI scraping error: {str(e)[:80]}")

    print(f"      → {new_count} PDF baru dari Bank Indonesia")
    return new_count

# ============================================================
# SCRAPER 3: Kemenkeu — APBN & Publikasi Fiskal
# ============================================================
def scrape_kemenkeu(registry: set) -> int:
    """Scrape PDF publikasi dari Kemenkeu."""
    print("   💰 Scraping Kemenkeu...")
    new_count  = 0
    page_url   = "https://www.kemenkeu.go.id/publikasi/berita/"

    try:
        resp = requests.get(page_url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(resp.text, "lxml")

        for link in soup.find_all("a", href=True):
            href = link["href"]
            if href.lower().endswith(".pdf"):
                if not href.startswith("http"):
                    href = "https://www.kemenkeu.go.id" + href

                filename = "kemenkeu_" + hashlib.md5(href.encode()).hexdigest()[:8] + ".pdf"
                if download_pdf(href, filename, registry):
                    new_count += 1
                    registry.add(filename)
                    time.sleep(1)

                if new_count >= 5:
                    break

    except Exception as e:
        print(f"      ❌ Kemenkeu scraping error: {str(e)[:80]}")

    print(f"      → {new_count} PDF baru dari Kemenkeu")
    return new_count

# ============================================================
# FUNGSI UTAMA: Jalankan semua scraper + re-ingest ke ChromaDB
# ============================================================
def run_scraping_pipeline():
    """
    Fungsi ini yang dipanggil APScheduler setiap 24 jam.
    1. Scrape semua sumber
    2. Kalau ada PDF baru → re-ingest ke ChromaDB
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n{'='*55}")
    print(f"🔄 Auto-Scraping dimulai: {timestamp}")
    print(f"{'='*55}")

    registry  = load_registry()
    total_new = 0

    total_new += scrape_bps(registry)
    total_new += scrape_bi(registry)
    total_new += scrape_kemenkeu(registry)

    print(f"\n📥 Total PDF baru: {total_new}")

    if total_new > 0:
        print("⚡ Menjalankan ingestion pipeline untuk PDF baru...")
        stats = ingest_pdfs(PDF_PATH, collection, embeddings)
        print(f"✅ ChromaDB diupdate: {collection.count()} dokumen total")
    else:
        print("✅ Tidak ada PDF baru — ChromaDB tetap up-to-date")

    # Catat ke log
    log_path = f"{BASE_DIR}/logs/scraping_log.txt"
    with open(log_path, "a") as f:
        f.write(f"{timestamp} | PDF baru: {total_new} | "
                f"Total ChromaDB: {collection.count()}\n")

    print(f"📝 Log disimpan ke: {log_path}")
    print(f"{'='*55}\n")

# ============================================================
# SETUP APSCHEDULER
# ============================================================
print("⚙️  Menyiapkan scheduler...")

scheduler = BackgroundScheduler()
scheduler.add_job(
    func=run_scraping_pipeline,
    trigger=IntervalTrigger(hours=24),
    id="pdf_scraping",
    name="Auto PDF Scraping",
    replace_existing=True,
)
scheduler.start()

print("✅ Scheduler aktif!")
print(f"   Interval    : setiap 24 jam")
print(f"   Job berikutnya: {scheduler.get_jobs()[0].next_run_time}")
print(f"\n🚀 Menjalankan scraping pertama kali sekarang...")

# Jalankan sekali langsung saat cell ini dirun
run_scraping_pipeline()